# Session 4 - Preprocessing Assignment

Insurance dataset.

The assignment is to apply the same preprocessing that we did on the Titanic dataset in session 3.
So I copied the project of session 3 and I changed only `config/config.py` (this was the idea of the config file).
The three functions of session 3 did not change, and I added the steps of the workshop
(missing values, duplicates, category, describe, outliers) as new functions inside `preprocessing.py`.

Here I only import the functions and try them, exactly like `main.py` does but without the menu.

Project structure:

```
assignment/
    config/
        __init__.py
        config.py
    data/
        raw/
            insurance.csv
        processed/
            insurance_clean.csv   (created at the end)
    preprocessing.py
    main.py
```

In [1]:
import pandas as pd

from config import DATA_PATH, PROCESSED_PATH, COLS_TO_DROP, CATEGORY_LIMIT, CATEGORY_COLS, NUMERIC_COLS
from preprocessing import (
    Read_data_file,
    Drop_unnecessary_features,
    Check_data_type,
    Check_missing_values,
    Remove_duplicates,
    Convert_to_category,
    Find_outliers,
    Save_data,
)

pd.set_option("display.max_columns", None)

## 1 - Read the dataset

`Read_data_file(file_path)` reads the csv and returns a DataFrame.
If the path is wrong it does not crash, it prints a short message and returns `None`.

In [2]:
df = Read_data_file(DATA_PATH)
df.head()

[OK] The file was read : 1338 rows and 7 columns.


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### Trying the bad cases

Same as session 3, the function must not crash if the path is wrong, it prints a small message.

In [3]:
# the file does not exist
Read_data_file("data/raw/insurance_wrong_name.csv")

[ERROR] The file was not found: data/raw/insurance_wrong_name.csv
        -> check the name and the folder of the file.


In [4]:
# the path is a folder and not a file
Read_data_file("data/raw")

[ERROR] This path is a folder not a file: data/raw


In [5]:
# the path is not even a text
Read_data_file(None)

[ERROR] The path is not valid, please write the path as a text.


## 2 - Remove the unnecessary features

In Titanic we removed `PassengerId`, `Name` and `Ticket` because they were just a counter, a text, and a code.

In the insurance data there is **no id column and no name column**, the 7 columns look useful
(`age`, `sex`, `bmi`, `children`, `smoker`, `region`, `charges`).
So the list `COLS_TO_DROP` in `config.py` is empty, and the function must work also with an empty list.

In [6]:
print(COLS_TO_DROP)
df = Drop_unnecessary_features(df, COLS_TO_DROP)
df.head()

[]
[INFO] The list of columns is empty, nothing was removed.


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [7]:
# just to try that the function still works, I do not save the result in df
Drop_unnecessary_features(df, ["region"]).head()

[OK] 1 column(s) removed: ['region']
     the data now has 6 columns.


,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520


In [8]:
# if I give a column that does not exist the function does not crash, it just warns me
Drop_unnecessary_features(df, ["PassengerId", "id"]).head()

[WARNING] These columns are not in the data so I ignored them: ['PassengerId', 'id']
[INFO] No column was removed.


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 3 - Inspect the dataset

`Check_data_type(df)` is the small data quality report of session 3.
For every column it gives the datatype, the number of unique values, the missing values
and a small note if the column looks like a category.

In [9]:
report = Check_data_type(df, CATEGORY_LIMIT)
report

Column,age,sex,bmi,children,smoker,region,charges
Dtype,int64,str,float64,int64,str,str,float64
N_unique,47,2,548,6,2,4,1337
N_missing,0,0,0,0,0,0,0
Missing_%,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Looks_like,numeric,category,numeric,category,category,category,numeric


### Reading the report

- There is **no missing values** in any column (all the `Missing_%` are 0), this is different from Titanic where `Age` and `Cabin` had a lot of missing.
- The columns that look like categories are: `sex` (2 values), `smoker` (2 values), `region` (4 values) and `children` (6 values).
  `children` is a number (`int64`) but it is a small count from 0 to 5, so I will treat it like a category.
- `age`, `bmi` and `charges` are the numeric columns.
- `charges` has **1337** unique values but the data has **1338** rows, so maybe there is one row that is repeated. I will check this in the duplicates step.

## 4 - Missing values

In the workshop we wrote two lines (`null` and `ratio`) and we put them in a table.
Now they are inside `Check_missing_values(df)`.

In [10]:
Check_missing_values(df)

[OK] There is no missing values in the data.


,age,sex,bmi,children,smoker,region,charges
N_missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Missing_%,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The table is all zeros, so there is nothing to drop or to fill.
(In Titanic we had to use `dropna` on `Embarked`, here we do not need it.)

## 5 - Duplicated rows

From the report `charges` had 1337 unique values in 1338 rows, so I check the rows that are repeated.

In [11]:
# keep=False shows me the two copies of the repeated row
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [12]:
df = Remove_duplicates(df)
df.shape

[OK] 1 duplicated row(s) removed.
     the data now has 1337 rows.


(1337, 7)

It was the same person written two times (19 years old male from northwest), so I removed one copy.
Now the data has **1337** rows.

## 6 - Convert the categorical columns

In the workshop we converted **all** the columns to `category`, and after that we had to convert `Age` and `Fare` back to numbers.
Here I learned from that and I convert only the columns that looked like a category in the report.
The list `CATEGORY_COLS` is saved in `config.py`.

In [13]:
print(CATEGORY_COLS)
df = Convert_to_category(df, CATEGORY_COLS)
pd.DataFrame(df.dtypes, columns=["Dtype"]).T

['sex', 'smoker', 'region', 'children']
[OK] 4 column(s) converted to category: ['sex', 'smoker', 'region', 'children']


,age,sex,bmi,children,smoker,region,charges
Dtype,int64,category,float64,category,category,category,float64


In [14]:
# describe of the numeric columns
df.describe()

,age,bmi,charges
count,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,13279.121487
std,14.044333,6.100468,12110.359656
min,18.000000,15.960000,1121.873900
25%,27.000000,26.290000,4746.344000
50%,39.000000,30.400000,9386.161300
75%,51.000000,34.700000,16657.717450
max,64.000000,53.130000,63770.428010


In [15]:
# describe of the category columns
df.describe(include="category")

,sex,children,smoker,region
count,1337,1337,1337,1337
unique,2,6,2,4
top,male,0,no,southeast
freq,675,573,1063,364


### Reading the describe

- `age` goes from **18** to **64**, the mean is about 39.
- `bmi` mean is about **30.7**. A bmi more than 30 is obese, so the mean person in the data is a little obese.
- `charges` mean is about **13279** but the median (50%) is about **9386**, the mean is bigger than the median so the column is skewed to the right (some people pay a lot).
- `smoker`: the top value is `no` with 1063 rows, so only about 20% of the people are smokers.
- `region`: the 4 regions are almost balanced, `southeast` is the biggest one.
- `children`: most of the people have 0 children.

## 7 - Outliers (IQR rule)

The rule that we took in the session:

```
IQR         = Q3 - Q1
lower fence = Q1 - 1.5 * IQR
upper fence = Q3 + 1.5 * IQR
```

Every value outside the two fences is an outlier.
In the session we wrote `Q1 + 1.5 * IQR` for the upper fence by mistake, the correct one is `Q3`, so I fixed it in the function.
`Find_outliers(df, num_cols)` does not remove anything, it only gives me a small report.

In [16]:
outliers_report = Find_outliers(df, NUMERIC_COLS)
outliers_report

[OK] The outliers were checked in 3 column(s).


,Q1,Q3,IQR,Lower_fence,Upper_fence,N_low_outliers,N_high_outliers,Outliers_%
Column,,,,,,,,
age,27.00,51.00,24.00,-9.00,87.00,0,0,0.00
bmi,26.29,34.70,8.41,13.67,47.32,0,9,0.67
charges,4746.34,16657.72,11911.37,-13120.72,34524.78,0,139,10.40


In [17]:
# let me see the bmi outliers
df[df["bmi"] > outliers_report.loc["bmi", "Upper_fence"]]

,age,sex,bmi,children,smoker,region,charges
116,58,male,49.06,0,no,southeast,11381.32540
286,46,female,48.07,2,no,northeast,9432.92530
401,47,male,47.52,1,no,southeast,8083.91980
543,54,female,47.41,0,yes,southeast,63770.42801
846,23,male,50.38,1,no,southeast,2438.05520
859,37,female,47.60,2,yes,southwest,46113.51100
1046,22,male,52.58,1,yes,southeast,44501.39820
1087,52,male,47.74,1,no,southeast,9748.91060
1316,18,male,53.13,0,no,southeast,1163.46270


In [18]:
# the charges outliers, are they smokers ?
high_charges = df[df["charges"] > outliers_report.loc["charges", "Upper_fence"]]
high_charges["smoker"].value_counts()

smoker
yes    136
no       3
Name: count, dtype: int64

### Reading the outliers report

- `age` has **no outliers**, the values are between 18 and 64 and this is normal.
- `bmi` has **9** high outliers, all of them are more than 47. It is a very high bmi but they are real people, so I keep them.
- `charges` has **139** high outliers (about **10%** of the data) and this is a lot.
  When I looked at them, **136** of the 139 are smokers. So they are not errors, it is the smokers who pay a lot of money.
  I will **not remove them** because I will lose the most important information in the data (smoker -> high charges).
  Later in the ML part maybe we handle them in another way (like a log transformation), but this is not the goal of this session.

## 8 - Save the clean data

The last step is to save the data after the preprocessing in `data/processed/` so I can use it in the next sessions.

In [19]:
Save_data(df, PROCESSED_PATH)

[OK] The clean data was saved : D:\03_DEPI\1_Technincal\00_Project\CA_AIS2_G-_Ml\src\DA\session-4\assignment\data\processed\insurance_clean.csv
     1337 rows and 7 columns.


True

## Summary

What I did on the insurance dataset (the same steps of the Titanic):

| Step | Titanic (session 3) | Insurance (session 4) |
|---|---|---|
| Read the file | 891 rows, 12 columns | 1338 rows, 7 columns |
| Unnecessary features | removed `PassengerId`, `Name`, `Ticket` | nothing to remove |
| Missing values | `Age` 19.9%, `Cabin` 77%, `Embarked` 2 rows | 0 missing |
| Duplicated rows | - | 1 row removed -> 1337 rows |
| Category columns | `Survived`, `Pclass`, `Sex`, `SibSp`, `Parch`, `Embarked` | `sex`, `smoker`, `region`, `children` |
| Outliers (IQR) | `Age` and `Fare` | `bmi` 9 rows, `charges` 139 rows (kept) |

The insurance data is much cleaner than Titanic: no missing values, only one duplicated row, and the outliers of `charges` are real (the smokers).